### Data Ingestion & Preprocessing

In [1]:
# train_item_knn.py
from surprise import Dataset, Reader, KNNBasic
import pickle

# 1. Load MovieLens (or Letterboxd-scraped) community ratings
reader = Reader(line_format="user item rating timestamp", sep=",", skip_lines=1)
data   = Dataset.load_from_file("ml-32m/ratings.csv", reader=reader)


### 2. Build full trainset & train item-based KNN

In [2]:
# 2. Build full trainset & train item-based KNN
trainset   = data.build_full_trainset()
sim_options = { "name": "cosine", "user_based": False }  # <-- item-based
algo       = KNNBasic(sim_options=sim_options, n_jobs=-1)
algo.fit(trainset)

: 

### 3. Persist the trained model

In [ ]:
with open("models/item_knn.pkl", "wb") as f:
    pickle.dump(algo, f)

print("Item–item KNN trained and saved.")

## Predict for a new user without retraining

In [ ]:
# predict_item_knn.py
import pickle
import pandas as pd

def recommend_for_user(
    enriched_user_csv: str,
    movies_csv: str,
    model_path: str,
    K: int = 40,
    N: int = 10
):
    # 1) Load the trained item-based model
    with open(model_path, "rb") as f:
        algo = pickle.load(f)

    # 2) Read community movie metadata
    movies = pd.read_csv(movies_csv)  # cols: movieId,title,...

    # 3) Read the *enriched* Letterboxd CSV
    #    (must have columns: tmdb_id, rating)
    user_df = pd.read_csv(enriched_user_csv)
    #    merge to get MovieLens movieId
    links   = pd.read_csv("ml-25m/links.csv")  # movieId,tmdbId
    merged  = user_df.merge(links, left_on="tmdb_id", right_on="tmdbId")
    user_ratings = dict(zip(merged.movieId, merged.rating))

    trainset   = algo.trainset
    sim_matrix = algo.sim  # 2D numpy array of shape [n_items, n_items]
    raw2inner  = trainset._raw2inner_id_items
    inner2raw  = trainset._inner2raw_id_items

    scores   = {}
    sim_sums = {}

    # 4) For each movie *j* the user rated, pull its top-K neighbors
    for movieId_j, r_j in user_ratings.items():
        if movieId_j not in raw2inner:
            continue
        inner_j = raw2inner[movieId_j]
        # neighbors = list of inner IDs
        neighbors = algo.get_neighbors(inner_j, k=K)

        for inner_i in neighbors:
            raw_i = inner2raw[inner_i]
            if raw_i in user_ratings:
                continue  # skip movies the user already rated
            sim_ij    = sim_matrix[inner_j, inner_i]
            scores.setdefault(raw_i, 0.0)
            sim_sums.setdefault(raw_i, 0.0)

            scores[raw_i]   += sim_ij * r_j
            sim_sums[raw_i] += abs(sim_ij)

    # 5) Build final predictions: weighted average
    preds = []
    for movieId_i, num in scores.items():
        den = sim_sums[movieId_i]
        if den > 0:
            est = num / den
            preds.append((movieId_i, est))

    # 6) Sort & pick top-N
    top_n = sorted(preds, key=lambda x: x[1], reverse=True)[:N]
    # map back to titles
    results = [
        ( movies.loc[movies.movieId==mid, "title"].iloc[0],
          round(score, 2) )
        for mid, score in top_n
    ]
    return results

if __name__ == "__main__":
    recs = recommend_for_user(
        enriched_user_csv="letterboxd_with_tmdb.csv",
        movies_csv="ml-25m/movies.csv",
        model_path="models/item_knn.pkl",
        K=50,
        N=10
    )
    print("Top 10 recommendations:")
    for title, score in recs:
        print(f"  • {title} — {score}★")


### Testing for enriching user letterboxed data with tmdbid's

In [ ]:
# from add_tmdb_ids import enrich_letterboxd_csv

# user_df = pd.read_csv("alex_data/ratings.csv")
# input_csv  = "alex_data/ratings.csv"
# output_csv = "alex_with_tmdb.csv"
# enrich_letterboxd_csv(input_csv, output_csv, sleep_between=0.25)
# print("Done enriching CSV!")

# user_df2 = pd.read_csv("alex_with_tmdb.csv")
# user_df2